In [1]:
import pyexasol
import configparser

In [2]:
config = configparser.ConfigParser()
config.read('C:\\Users\\svi02\\.spyder-py3\\ExasolDET.ini')

['C:\\Users\\svi02\\.spyder-py3\\ExasolDET.ini']

In [3]:
dsn=config['exasolDET']['dsn']
user=config['exasolDET']['user']
pwd=config['exasolDET']['pwd']
schema=config['exasolDET']['schema']

In [4]:
connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema)

In [5]:
# Column names
raw_data_col = ['HRSHotelNumber', 'HotelName', 'HotelChain', 'Street', 'ZipCode',
       'City', 'Country', 'Phone', 'Fax', 'Hotel/ChainContactName',
       'Hotel/ChainContactE-Mail', 'Hotel/ChainContactPhone', 'Priority',
       'RoomCategory', 'RateType', 'LRA/NLRA', 'ValidFrom',
       'ValidTo(including)', 'ValidWeekdays', 'PriceSGLperNight',
       'PriceDBLperNight', 'PriceDBLSingleUseperNight', 'Breakfastincluded',
       'BreakfastPriceperPerson', 'Currency', 'GrossPrice', 'LodgingTax',
       'StateTax', 'CityTax', 'VAT', 'ServiceTax', 'Occupancy/VisitorTax',
       'OtherTax', 'MinimumStay', 'ReservationType',
       'FreeCancellationDeadline(Days)', 'FreeCancellationDeadline(Time)',
       'RateDescription', 'Parkingincluded', 'ParkingPriceper24h',
       'Wi-Fiincluded', 'Wi-FiPriceper24h', 'Internetincluded',
       'InternetPriceper24h', 'BottleofWaterincluded', 'BottleofWaterPrice',
       'AirportTransferincluded', 'AirportTransferPrice',
       'OfficeTransferincluded', 'OfficeTransferPrice',
       'FitnessRoomUsageincluded', 'FitnessRoomUsagePrice',
       'Service1Description', 'Service1included', 'Service1Price',
       'Service2Description', 'Service2inluded', 'Service2Price',
       'Service3Description', 'Service3included', 'Service3Price',
       'Service4Description', 'Service4included', 'Service4Price',
       'Hotelstatus', 'Ratestatus', 'Locked']


In [6]:
# reading excel
import pandas as pd
filename = input("Enter the file name: ")
raw_data = pd.read_excel('C:\\Users\\svi02\\Documents\\RAW_DATA\\data\\'+filename+'.xls'
                         , sheet_name = 'Excel List'
                         , skiprows = [0,2]
                         #, usecols = raw_data_col
                        )
len(raw_data.columns)

Enter the file name: vw_1


67

In [7]:
import re
colnames_raw_data = raw_data.columns
len_raw_data_col = len(colnames_raw_data)

if len_raw_data_col == len(raw_data_col):
    # Strip white spaces
    colnames_raw_data = [col.strip() for col in colnames_raw_data]
    # Replace special characters, spaces and new lines
    colnames_raw_data = [re.sub('[*?\n\s]+', '', col) for col in colnames_raw_data]
    raw_data.columns = colnames_raw_data
    # Trimming the rows
    raw_data.replace('(^\s+|\s+$)', '', regex=True, inplace=True) 
    raw_data.head()
    
else:
    print("Columns count mismatch")
#raw_data_col = raw_data.columns
raw_data.shape

(8061, 67)

In [8]:
# Exporting normalised Corsa Data to csv
raw_data.to_csv('C:\\Users\\svi02\\Documents\\RAW_DATA\\data\\exasol_raw_data_import.csv'
               , sep =';'
               , encoding = 'utf-8'
               , index = False)

In [9]:
#raw_data_obj = raw_data.select_dtypes(['object'])
#raw_data_obj.columns
#raw_data_obj = raw_data_obj.apply(lambda x: x.str.strip())

In [10]:
# Truncate temp table
connect.execute("TRUNCATE TABLE temp.directory_test1")
import_query_location = 'C:\\Users\\svi02\\Documents\\RAW_DATA\\sql\\import_data_exasol_csv.txt' 
import_query_open = open(import_query_location, 'r')
import_query_read = import_query_open.read()
#connect.execute(import_query_read)
connect.import_from_file('C:\\Users\\svi02\\Documents\\RAW_DATA\\data\\exasol_raw_data_import.csv', 'directory_test1', import_params={'skip': 1, 'column_separator': ';'})

In [12]:
# Directory script execution
main_choice = input("Please enter Directory or Fair audit: ")

if main_choice == "Directory":

    choice = input("Please enter Siemens or VW: ")
    if choice == "Siemens":

        import_directory_script = 'C:\\Users\\svi02\\Documents\\RAW_DATA\\sql\\Siemens_Main_Directory_Script_Exasol_10052019.sql'

        dir_script_open = open(import_directory_script, 'r')
        dir_script_read = dir_script_open.read()
        #file_read
        directory_data = connect.execute(dir_script_read)

        # Importing data into a DataFrame
        directory_siemens = pd.DataFrame()
        a=[]
        for row in directory_data:
            a.append(row)
        #print(len(a))
        directory_siemens = pd.DataFrame(a)
        directory_siemens.columns = ['HRSHOTELNUMBER','HOTELNAME','HOTELCHAIN','STREET','ZIPCODE','CITY','COUNTRY','PHONE','FAX','Hotel/ChainContactName','Hotel/ChainContactE-Mail',
        'Hotel/ChainContactPhone','RATETYPE','ROOMCATEGORY','LRA/NLRA','BREAKFASTINCLUDED','BREAKFASTPRICEPERPERSON','CURRENCY','GROSSPRICE','LODGINGTAX','STATETAX',
        'CITYTAX','VAT','SERVICETAX','Occupancy/VisitorTax','OTHERTAX','FreeCancellationDeadline(Days)','FreeCancellationDeadline(Time)','PARKINGINCLUDED','PARKINGPRICEPER24H',
        'Wi-Fiincluded','Wi-FiPriceper24h','INTERNETINCLUDED','INTERNETPRICEPER24H','BOTTLEOFWATERINCLUDED','BOTTLEOFWATERPRICE','AIRPORTTRANSFERINCLUDED','AIRPORTTRANSFERPRICE',
        'OFFICETRANSFERINCLUDED','OFFICETRANSFERPRICE','FITNESSROOMUSAGEINCLUDED','FITNESSROOMUSAGEPRICE','S1_START_DATE','S1_END_DATE','S1_SGL_RATE','S1_DBL_RATE',
        'S2_START_DATE','S2_END_DATE','S2_SGL_RATE','S2_DBL_RATE','S3_START_DATE','S3_END_DATE','S3_SGL_RATE','S3_DBL_RATE','S4_START_DATE','S4_END_DATE','S4_SGL_RATE',
        'S4_DBL_RATE','S5_START_DATE','S5_END_DATE','S5_SGL_RATE','S5_DBL_RATE','S6_START_DATE','S6_END_DATE','S6_SGL_RATE','S6_DBL_RATE','S7_START_DATE','S7_END_DATE',
        'S7_SGL_RATE','S7_DBL_RATE','S8_START_DATE','S8_END_DATE','S8_SGL_RATE','S8_DBL_RATE','S9_START_DATE','S9_END_DATE','S9_SGL_RATE','S9_DBL_RATE','S10_START_DATE',
        'S10_END_DATE','S10_SGL_RATE','S10_DBL_RATE','S11_START_DATE','S11_END_DATE','S11_SGL_RATE','S11_DBL_RATE','S12_SGL_RATE','S12_START_DATE','S12_END_DATE',
        'S12_DBL_RATE','S13_START_DATE','S13_END_DATE','S13_SGL_RATE','S13_DBL_RATE','S14_START_DATE','S14_END_DATE','S14_SGL_RATE','S14_DBL_RATE','S15_START_DATE',
        'S15_END_DATE','S15_SGL_RATE','S15_DBL_RATE','S16_START_DATE','S16_END_DATE','S16_SGL_RATE','S16_DBL_RATE','S17_START_DATE','S17_END_DATE','S17_SGL_RATE','S17_DBL_RATE']

    elif choice == 'VW':
        import_directory_script = 'C:\\Users\\svi02\\Documents\\RAW_DATA\\sql\\Volkswagen_Directory_Script_Exasol_10052019.sql'

        dir_script_open = open(import_directory_script, 'r')
        dir_script_read = dir_script_open.read()
        #file_read
        directory_data = connect.execute(dir_script_read)

        # Importing data into a DataFrame
        directory_vw = pd.DataFrame()
        a=[]
        for row in directory_data:
            a.append(row)
        #print(len(a))
        directory_vw = pd.DataFrame(a)
        directory_vw.columns = ['HRSHOTELNUMBER', 'HOTELNAME', 'HOTELCHAIN', 'STREET', 'ZIPCODE', 'CITY', 'COUNTRY', 'PHONE', 'FAX', 'Hotel/ChainContactName',
    'Hotel/ChainContactE-Mail', 'Hotel/ChainContactPhone', 'RATETYPE', 'LRA/NLRA', 'BREAKFASTINCLUDED', 'BREAKFASTPRICEPERPERSON',
    'CURRENCY', 'GROSSPRICE', 'LODGINGTAX', 'STATETAX', 'CITYTAX', 'VAT', 'SERVICETAX', 'Occupancy/VisitorTax', 'OTHERTAX', 'FreeCancellationDeadline(Days)',
    'FreeCancellationDeadline(Time)', 'PARKINGINCLUDED', 'PARKINGPRICEPER24H', 'Wi-Fiincluded', 'Wi-FiPriceper24h', 'INTERNETINCLUDED', 'INTERNETPRICEPER24H',
    'BOTTLEOFWATERINCLUDED', 'BOTTLEOFWATERPRICE', 'AIRPORTTRANSFERINCLUDED', 'AIRPORTTRANSFERPRICE', 'OFFICETRANSFERINCLUDED', 'OFFICETRANSFERPRICE', 'FITNESSROOMUSAGEINCLUDED',
    'FITNESSROOMUSAGEPRICE', 'S1_START_DATE', 'S1_END_DATE', 'S1_STANDARD_SGL_RATE', 'S1_STANDARD_DBL_RATE', 'S1_SUPERIOR_SGL_RATE', 'S1_SUPERIOR_DBL_RATE','S2_START_DATE','S2_END_DATE',
    'S2_STANDARD_SGL_RATE', 'S2_STANDARD_DBL_RATE', 'S2_SUPERIOR_SGL_RATE', 'S2_SUPERIOR_DBL_RATE', 'S3_START_DATE', 'S3_END_DATE', 'S3_STANDARD_SGL_RATE', 'S3_STANDARD_DBL_RATE',
    'S3_SUPERIOR_SGL_RATE', 'S3_SUPERIOR_DBL_RATE', 'S4_START_DATE', 'S4_END_DATE', 'S4_STANDARD_SGL_RATE', 'S4_STANDARD_DBL_RATE', 'S4_SUPERIOR_SGL_RATE', 'S4_SUPERIOR_DBL_RATE',
    'S5_START_DATE', 'S5_END_DATE', 'S5_STANDARD_SGL_RATE', 'S5_STANDARD_DBL_RATE', 'S5_SUPERIOR_SGL_RATE', 'S5_SUPERIOR_DBL_RATE', 'S6_START_DATE', 'S6_END_DATE', 'S6_STANDARD_SGL_RATE',
    'S6_STANDARD_DBL_RATE', 'S6_SUPERIOR_SGL_RATE', 'S6_SUPERIOR_DBL_RATE', 'S7_START_DATE', 'S7_END_DATE', 'S7_STANDARD_SGL_RATE', 'S7_STANDARD_DBL_RATE', 'S7_SUPERIOR_SGL_RATE',
    'S7_SUPERIOR_DBL_RATE','S8_START_DATE', 'S8_END_DATE', 'S8_STANDARD_SGL_RATE', 'S8_STANDARD_DBL_RATE', 'S8_SUPERIOR_SGL_RATE', 'S8_SUPERIOR_DBL_RATE', 'S9_START_DATE', 'S9_END_DATE',
    'S9_STANDARD_SGL_RATE', 'S9_STANDARD_DBL_RATE', 'S9_SUPERIOR_SGL_RATE', 'S9_SUPERIOR_DBL_RATE', 'S10_START_DATE', 'S10_END_DATE','S10_STANDARD_SGL_RATE', 'S10_STANDARD_DBL_RATE',
    'S10_SUPERIOR_SGL_RATE','S10_SUPERIOR_DBL_RATE', 'S11_START_DATE', 'S11_END_DATE', 'S11_STANDARD_SGL_RATE', 'S11_STANDARD_DBL_RATE', 'S11_SUPERIOR_SGL_RATE', 'S11_SUPERIOR_DBL_RATE',
    'S12_START_DATE','S12_END_DATE', 'S12_STANDARD_SGL_RATE','S12_STANDARD_DBL_RATE', 'S12_SUPERIOR_SGL_RATE','S12_SUPERIOR_DBL_RATE','S13_START_DATE', 'S13_END_DATE', 'S13_STANDARD_SGL_RATE',
    'S13_STANDARD_DBL_RATE','S13_SUPERIOR_SGL_RATE', 'S13_SUPERIOR_DBL_RATE','S14_START_DATE','S14_END_DATE','S14_STANDARD_SGL_RATE','S14_STANDARD_DBL_RATE','S14_SUPERIOR_SGL_RATE','S14_SUPERIOR_DBL_RATE',
    'S15_START_DATE','S15_END_DATE','S15_STANDARD_SGL_RATE','S15_STANDARD_DBL_RATE','S15_SUPERIOR_SGL_RATE','S15_SUPERIOR_DBL_RATE','S16_START_DATE','S16_END_DATE','S16_STANDARD_SGL_RATE','S16_STANDARD_DBL_RATE',
    'S16_SUPERIOR_SGL_RATE','S16_SUPERIOR_DBL_RATE','S17_START_DATE','S17_END_DATE','S17_STANDARD_SGL_RATE','S17_STANDARD_DBL_RATE','S17_SUPERIOR_SGL_RATE','S17_SUPERIOR_DBL_RATE']
    else:
        print("Please enter 'Siemens' or 'VW'.....")
else:
    print("Running fair audit script")
    fair_choice = input("Please enter Siemens or VW: ")
    if fair_choice == "Siemens":
        import_directory_script = 'C:\\Users\\svi02\\Documents\\RAW_DATA\\sql\\Siemens_Fareaudit_Directory_Script_Exasol_10052019.sql'

        dir_script_open = open(import_directory_script, 'r')
        dir_script_read = dir_script_open.read()
        #file_read
        directory_data = connect.execute(dir_script_read)

        # Importing data into a DataFrame
        fair_siemens = pd.DataFrame()
        a=[]
        for row in directory_data:
            a.append(row)
        #print(len(a))
        fair_siemens = pd.DataFrame(a)
        fair_siemens.columns = ['HRSHOTELNumber','HotelName','HotelChain','Street','ZipCode','City','Country','Phone','Fax','Hotel/ChainContactName','Hotel/ChainContactE-Mail','Hotel/ChainContactPhone','RateType','LRA/NLRA','BREAKFASTINCLUDED',
'BREAKFASTPRICEPERPERSON','Currency','GrossPrice','LodgingTax','StateTax','CityTax','VAT','ServiceTax','Occupancy/VisitorTax','OtherTax','FreeCancellationDeadline(Days)','FreeCancellationDeadline(Time)','Parkingincluded',
'ParkingPriceper24h','Wi-Fiincluded','Wi-FiPriceper24h','Internetincluded','InternetPriceper24h','BottleofWaterincluded','BottleofWaterPrice','AirportTransferincluded','AirportTransferPrice','OfficeTransferincluded',
'OfficeTransferPrice','FitnessRoomUsageincluded','FitnessRoomUsagePrice','S1_Start_Date','S1_End_Date','S1_Standard_Sgl_Rate','S1_Standard_Dbl_Rate','S1_Superior_Sgl_Rate','S1_Superior_Dbl_Rate','S2_Start_Date','S2_End_Date',
'S2_Standard_Sgl_Rate','S2_Standard_Dbl_Rate','S2_Superior_Sgl_Rate','S2_Superior_Dbl_Rate','S3_Start_Date','S3_End_Date','S3_Standard_Sgl_Rate','S3_Standard_Dbl_Rate','S3_Superior_Sgl_Rate','S3_Superior_Dbl_Rate','S4_Start_Date',
'S4_End_Date','S4_Standard_Sgl_Rate','S4_Standard_Dbl_Rate','S4_Superior_Sgl_Rate','S4_Superior_Dbl_Rate','S5_Start_Date','S5_End_Date','S5_Standard_Sgl_Rate','S5_Standard_Dbl_Rate','S5_Superior_Sgl_Rate','S5_Superior_Dbl_Rate',
'S6_Start_Date','S6_End_Date','S6_Standard_Sgl_Rate','S6_Standard_Dbl_Rate','S6_Superior_Sgl_Rate','S6_Superior_Dbl_Rate','S7_Start_Date','S7_End_Date','S7_Standard_Sgl_Rate','S7_Standard_Dbl_Rate','S7_Superior_Sgl_Rate',
'S7_Superior_Dbl_Rate','S8_Start_Date','S8_End_Date','S8_Standard_Sgl_Rate','S8_Standard_Dbl_Rate','S8_Superior_Sgl_Rate','S8_Superior_Dbl_Rate','S9_Start_Date','S9_End_Date','S9_Standard_Sgl_Rate','S9_Standard_Dbl_Rate','S9_Superior_Sgl_Rate',
'S9_Superior_Dbl_Rate','S10_Start_Date','S10_End_Date','S10_Standard_Sgl_Rate','S10_Standard_Dbl_Rate','S10_Superior_Sgl_Rate','S10_Superior_Dbl_Rate','S11_Start_Date','S11_End_Date','S11_Standard_Sgl_Rate','S11_Standard_Dbl_Rate',
'S11_Superior_Sgl_Rate','S11_Superior_Dbl_Rate','S12_Start_Date','S12_End_Date','S12_Standard_Sgl_Rate','S12_Standard_Dbl_Rate','S12_Superior_Sgl_Rate','S12_Superior_Dbl_Rate','S13_Start_Date','S13_End_Date','S13_Standard_Sgl_Rate',
'S13_Standard_Dbl_Rate','S13_Superior_Sgl_Rate','S13_Superior_Dbl_Rate','S14_Start_Date','S14_End_Date','S14_Standard_Sgl_Rate','S14_Standard_Dbl_Rate','S14_Superior_Sgl_Rate','S14_Superior_Dbl_Rate','S15_Start_Date','S15_End_Date',
'S15_Standard_Sgl_Rate','S15_Standard_Dbl_Rate','S15_Superior_Sgl_Rate','S15_Superior_Dbl_Rate','S16_Start_Date','S16_End_Date','S16_Standard_Sgl_Rate','S16_Standard_Dbl_Rate','S16_Superior_Sgl_Rate','S16_Superior_Dbl_Rate','HRSHOTELNUMBER_TF_BO',
'S1_BLACKOUT_START_DATE','S1_BLACKOUT_END_DATE','S2_BLACKOUT_START_DATE','S2_BLACKOUT_END_DATE','S3_BLACKOUT_START_DATE','S3_BLACKOUT_END_DATE','S4_BLACKOUT_START_DATE','S4_BLACKOUT_END_DATE','S5_BLACKOUT_START_DATE','S5_BLACKOUT_END_DATE',
'S6_BLACKOUT_START_DATE','S6_BLACKOUT_END_DATE','S7_BLACKOUT_START_DATE','S7_BLACKOUT_END_DATE','S8_BLACKOUT_START_DATE','S8_BLACKOUT_END_DATE','S9_BLACKOUT_START_DATE','S9_BLACKOUT_END_DATE','S10_BLACKOUT_START_DATE',
'S10_BLACKOUT_END_DATE','S11_BLACKOUT_START_DATE','S11_BLACKOUT_END_DATE','S12_BLACKOUT_START_DATE','S12_BLACKOUT_END_DATE','S13_BLACKOUT_START_DATE','S13_BLACKOUT_END_DATE','S14_BLACKOUT_START_DATE','S14_BLACKOUT_END_DATE',
'S15_BLACKOUT_START_DATE','S15_BLACKOUT_END_DATE']
        
    elif fair_choice == "VW":
        import_directory_script = 'C:\\Users\\svi02\\Documents\\RAW_DATA\\sql\\Volkswagen_Fair_audit_Exasol_10052019.sql'

        dir_script_open = open(import_directory_script, 'r')
        dir_script_read = dir_script_open.read()
        #file_read
        directory_data = connect.execute(dir_script_read)

        # Importing data into a DataFrame
        fair_vw = pd.DataFrame()
        a=[]
        for row in directory_data:
            a.append(row)
        #print(len(a))
        fair_vw = pd.DataFrame(a)
        fair_vw.columns = ['HRSHOTELNUMBER', 'HOTELNAME', 'HOTELCHAIN', 'STREET', 'ZIPCODE', 'CITY', 'COUNTRY', 'PHONE', 'FAX', 'Hotel/ChainContactName', 'Hotel/ChainContactE-Mail', 'Hotel/ChainContactPhone', 'RATETYPE', 'LRA/NLRA', 'BREAKFASTINCLUDED',
 'BREAKFASTPRICEPERPERSON', 'CURRENCY', 'GROSSPRICE', 'LODGINGTAX', 'STATETAX', 'CITYTAX', 'VAT', 'SERVICETAX', 'Occupancy/VisitorTax', 'OTHERTAX', 'FreeCancellationDeadline(Days)', 'FreeCancellationDeadline(Time)',
 'PARKINGINCLUDED', 'PARKINGPRICEPER24H', 'Wi-Fiincluded', 'Wi-FiPriceper24h', 'INTERNETINCLUDED', 'INTERNETPRICEPER24H', 'BOTTLEOFWATERINCLUDED', 'BOTTLEOFWATERPRICE', 'AIRPORTTRANSFERINCLUDED', 'AIRPORTTRANSFERPRICE',
 'OFFICETRANSFERINCLUDED', 'OFFICETRANSFERPRICE', 'FITNESSROOMUSAGEINCLUDED', 'FITNESSROOMUSAGEPRICE', 'S1_START_DATE', 'S1_END_DATE', 'S1_STANDARD_SGL_RATE', 'S1_STANDARD_DBL_RATE', 'S1_SUPERIOR_SGL_RATE',
 'S1_SUPERIOR_DBL_RATE', 'S2_START_DATE', 'S2_END_DATE', 'S2_STANDARD_SGL_RATE', 'S2_STANDARD_DBL_RATE', 'S2_SUPERIOR_SGL_RATE', 'S2_SUPERIOR_DBL_RATE', 'S3_START_DATE', 'S3_END_DATE', 'S3_STANDARD_SGL_RATE',
 'S3_STANDARD_DBL_RATE', 'S3_SUPERIOR_SGL_RATE', 'S3_SUPERIOR_DBL_RATE', 'S4_START_DATE', 'S4_END_DATE', 'S4_STANDARD_SGL_RATE', 'S4_STANDARD_DBL_RATE', 'S4_SUPERIOR_SGL_RATE', 'S4_SUPERIOR_DBL_RATE', 'S5_START_DATE',
 'S5_END_DATE', 'S5_STANDARD_SGL_RATE', 'S5_STANDARD_DBL_RATE', 'S5_SUPERIOR_SGL_RATE', 'S5_SUPERIOR_DBL_RATE', 'S6_START_DATE', 'S6_END_DATE', 'S6_STANDARD_SGL_RATE', 'S6_STANDARD_DBL_RATE', 'S6_SUPERIOR_SGL_RATE',
 'S6_SUPERIOR_DBL_RATE', 'S7_START_DATE', 'S7_END_DATE', 'S7_STANDARD_SGL_RATE', 'S7_STANDARD_DBL_RATE', 'S7_SUPERIOR_SGL_RATE', 'S7_SUPERIOR_DBL_RATE', 'S8_START_DATE', 'S8_END_DATE', 'S8_STANDARD_SGL_RATE',
 'S8_STANDARD_DBL_RATE', 'S8_SUPERIOR_SGL_RATE', 'S8_SUPERIOR_DBL_RATE', 'S9_START_DATE', 'S9_END_DATE', 'S9_STANDARD_SGL_RATE', 'S9_STANDARD_DBL_RATE', 'S9_SUPERIOR_SGL_RATE', 'S9_SUPERIOR_DBL_RATE', 'S10_START_DATE',
 'S10_END_DATE', 'S10_STANDARD_SGL_RATE', 'S10_STANDARD_DBL_RATE', 'S10_SUPERIOR_SGL_RATE', 'S10_SUPERIOR_DBL_RATE', 'S11_START_DATE', 'S11_END_DATE', 'S11_STANDARD_SGL_RATE', 'S11_STANDARD_DBL_RATE', 'S11_SUPERIOR_SGL_RATE',
 'S11_SUPERIOR_DBL_RATE', 'S12_START_DATE', 'S12_END_DATE', 'S12_STANDARD_SGL_RATE', 'S12_STANDARD_DBL_RATE', 'S12_SUPERIOR_SGL_RATE', 'S12_SUPERIOR_DBL_RATE', 'S13_START_DATE', 'S13_END_DATE', 'S13_STANDARD_SGL_RATE',
 'S13_STANDARD_DBL_RATE', 'S13_SUPERIOR_SGL_RATE', 'S13_SUPERIOR_DBL_RATE', 'S14_START_DATE', 'S14_END_DATE', 'S14_STANDARD_SGL_RATE', 'S14_STANDARD_DBL_RATE', 'S14_SUPERIOR_SGL_RATE', 'S14_SUPERIOR_DBL_RATE', 'S15_START_DATE',
 'S15_END_DATE', 'S15_STANDARD_SGL_RATE', 'S15_STANDARD_DBL_RATE', 'S15_SUPERIOR_SGL_RATE', 'S15_SUPERIOR_DBL_RATE', 'S16_START_DATE', 'S16_END_DATE', 'S16_STANDARD_SGL_RATE', 'S16_STANDARD_DBL_RATE', 'S16_SUPERIOR_SGL_RATE',
 'S16_SUPERIOR_DBL_RATE', 'S17_START_DATE', 'S17_END_DATE', 'S17_STANDARD_SGL_RATE', 'S17_STANDARD_DBL_RATE', 'S17_SUPERIOR_SGL_RATE', 'S17_SUPERIOR_DBL_RATE']  
 
    else:
        print("Wrong choice, enter again")

Please enter Directory or Fair audit: Fair
Running fair audit script
Please enter Siemens or VW: VW


In [54]:
fair_[fair_siemens.HRSHOTELNumber == 27, ['HRSHOTELNumber','HotelName']]

TypeError: 'Series' objects are mutable, thus they cannot be hashed

In [23]:
### OPTIONAL
###################### SIEMENS
## Changing str(date) columns to DATE format


dir_date_list = ['S1_START_DATE','S1_END_DATE','S2_START_DATE','S2_END_DATE','S3_START_DATE','S3_END_DATE','S4_START_DATE','S4_END_DATE','S5_START_DATE','S5_END_DATE','S6_START_DATE','S6_END_DATE','S7_START_DATE','S7_END_DATE','S8_START_DATE',
'S8_END_DATE','S9_START_DATE','S9_END_DATE','S10_START_DATE','S10_END_DATE','S11_START_DATE','S11_END_DATE','S12_START_DATE','S12_END_DATE','S13_START_DATE','S13_END_DATE','S14_START_DATE','S14_END_DATE','S15_START_DATE',
'S15_END_DATE','S16_START_DATE','S16_END_DATE','S17_START_DATE','S17_END_DATE']

fair_date_list = ['S1_Start_Date','S1_End_Date','S2_Start_Date','S2_End_Date','S3_Start_Date','S3_End_Date','S4_Start_Date','S4_End_Date','S5_Start_Date','S5_End_Date',
'S6_Start_Date','S6_End_Date','S7_Start_Date','S7_End_Date','S8_Start_Date','S8_End_Date','S9_Start_Date','S9_End_Date','S10_Start_Date','S10_End_Date',
'S11_Start_Date','S11_End_Date','S12_Start_Date','S12_End_Date','S13_Start_Date','S13_End_Date','S14_Start_Date','S14_End_Date','S15_Start_Date','S15_End_Date',
'S16_Start_Date','S16_End_Date','S1_BLACKOUT_START_DATE','S1_BLACKOUT_END_DATE','S2_BLACKOUT_START_DATE','S2_BLACKOUT_END_DATE','S3_BLACKOUT_START_DATE','S3_BLACKOUT_END_DATE',
'S4_BLACKOUT_START_DATE','S4_BLACKOUT_END_DATE','S5_BLACKOUT_START_DATE','S5_BLACKOUT_END_DATE','S6_BLACKOUT_START_DATE','S6_BLACKOUT_END_DATE',
'S7_BLACKOUT_START_DATE','S7_BLACKOUT_END_DATE','S8_BLACKOUT_START_DATE','S8_BLACKOUT_END_DATE','S9_BLACKOUT_START_DATE','S9_BLACKOUT_END_DATE',
'S10_BLACKOUT_START_DATE','S10_BLACKOUT_END_DATE','S11_BLACKOUT_START_DATE','S11_BLACKOUT_END_DATE','S12_BLACKOUT_START_DATE','S12_BLACKOUT_END_DATE',
'S13_BLACKOUT_START_DATE','S13_BLACKOUT_END_DATE','S14_BLACKOUT_START_DATE','S14_BLACKOUT_END_DATE','S15_BLACKOUT_START_DATE','S15_BLACKOUT_END_DATE']


for item in dir_date_list:
    directory_siemens[item] = pd.to_datetime(directory_siemens[item], format ='%Y-%m-%d')

for item in fair_date_list:
    fair_siemens[item] = pd.to_datetime(fair_siemens[item], format ='%Y-%m-%d')
    

In [41]:
### OPTIONAL
###################### VW
## Changing str(date) columns to DATE format
from datetime import datetime,date as dt
date_list = ['S1_START_DATE','S1_END_DATE','S2_START_DATE','S2_END_DATE','S3_START_DATE','S3_END_DATE','S4_START_DATE','S4_END_DATE','S5_START_DATE','S5_END_DATE','S6_START_DATE','S6_END_DATE','S7_START_DATE','S7_END_DATE','S8_START_DATE',
'S8_END_DATE','S9_START_DATE','S9_END_DATE','S10_START_DATE','S10_END_DATE','S11_START_DATE','S11_END_DATE','S12_START_DATE','S12_END_DATE','S13_START_DATE','S13_END_DATE','S14_START_DATE','S14_END_DATE','S15_START_DATE',
'S15_END_DATE','S16_START_DATE','S16_END_DATE','S17_START_DATE','S17_END_DATE']

for item in date_list:
   # datetime.strptime('Jun 1 2005  1:33PM', '%b %d %Y %I:%M%p')
    directory_vw[item] = pd.to_datetime(directory_vw[item], format ='%Y/%m/%d').dt.date
    

fair_vw[['S1_START_DATE','S1_END_DATE','S2_START_DATE','S2_END_DATE','S3_START_DATE','S3_END_DATE','S4_START_DATE','S4_END_DATE','S5_START_DATE','S5_END_DATE','S6_START_DATE','S6_END_DATE','S7_START_DATE','S7_END_DATE','S8_START_DATE',
'S8_END_DATE','S9_START_DATE','S9_END_DATE','S10_START_DATE','S10_END_DATE','S11_START_DATE','S11_END_DATE','S12_START_DATE','S12_END_DATE','S13_START_DATE','S13_END_DATE','S14_START_DATE','S14_END_DATE','S15_START_DATE',
'S15_END_DATE','S16_START_DATE','S16_END_DATE','S17_START_DATE','S17_END_DATE']]
    
for item in date_list:
    fair_vw[item] = pd.to_datetime(fair_vw[item], format ='%Y/%m/%d').dt.date
   
            


In [42]:
# DF to Excel for VOLKSWAGEN
import datetime as d
file = "C:\\Users\\svi02\\Documents\\RAW_DATA\\results\\VW_Directory_fair_audit_"+ str(d.datetime.now().strftime("%Y%m%d_%H%M%S"))+".xlsx" 
with pd.ExcelWriter(file) as writer: 
    directory_vw.to_excel(writer, sheet_name='Directory', header=True, encoding='utf-8', index=False, freeze_panes=(1,1))
    fair_vw.to_excel(writer, sheet_name='Fair_audit', header=True, encoding='utf-8', index=False, freeze_panes=(1,1))

In [25]:
# DF to Excel for SIEMENS
import datetime as d
file = "C:\\Users\\svi02\\Documents\\RAW_DATA\\results\\BearingPoint_Siemens_Directory_fair_audit_"+ str(d.datetime.now().strftime("%Y%m%d_%H%M%S"))+".xlsx" 
with pd.ExcelWriter(file) as writer: 
    directory_siemens.to_excel(writer, sheet_name='Directory', header=True, encoding='utf-8', index=False, freeze_panes=(1,1))
    fair_siemens.to_excel(writer, sheet_name='Fair_audit', header=True, encoding='utf-8', index=False, freeze_panes=(1,1))

In [40]:
directory_vw.S1_START_DATE

0      2020-01-01
1      2020-01-01
2      2020-01-01
3      2020-01-01
4      2020-01-01
          ...    
1556   2020-01-01
1557   2020-01-01
1558   2020-01-01
1559   2020-01-01
1560   2020-01-01
Name: S1_START_DATE, Length: 1561, dtype: datetime64[ns]